# Concept 3: Tool / Function Calling

Giving the model a menu of actions, not just words. This is the backbone of every AI agent and integration: you describe functions with schemas, the model decides whether to call one, and returns structured arguments instead of prose.

## The Request → Decide → Execute Loop
1. You send the user's message + list of available tools

2. The model responds: either plain text, or a tool call with structured arguments

3. Your code runs the real function with those arguments

4. You send the result back to the model, which produces the final answer

## Setup
This demo calls the real OpenAI API. In Colab, store your key using **Secrets** (the key icon in the left sidebar) under the name `OPENAI_API_KEY`, and enable notebook access for it.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

## Demo: Calling a Real Tool

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city", "unit"]
        }
    }
}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "I'm travelling to Pune tomorrow, what should I pack? Use celsius."}],
    tools=tools
)

call = response.choices[0].message.tool_calls[0]
print("Function called:", call.function.name)
print("Arguments:", call.function.arguments)

## What Just Happened
The prompt never mentioned weather directly — it was about packing for a trip. But the model figured out it needed weather data to answer well, and called `get_weather` on its own, filling in `city` and `unit` exactly matching our schema. That's a JSON string your code can `json.loads()` directly, no regex required.

## Try It Yourself
1. Change the prompt to something that shouldn't need the tool, e.g. `"Thanks, that's all!"`, and check whether `tool_calls` is empty
2. Try another indirect prompt, e.g. `"Should I carry an umbrella if I'm heading out in Tokyo today? Fahrenheit please."`, and see the model infer the same tool call
3. Add a second tool (e.g. `get_forecast`) and see which one the model picks for different prompts